# Decline curve analysis

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/olaflaitinen/wellcast/blob/main/notebooks/02_decline_curve_analysis.ipynb)

**Author:** Olaf Yunus Laitinen Imanov ([ORCID 0009-0006-5184-0810](https://orcid.org/0009-0006-5184-0810))  
**Affiliation:** Department of Mathematics and Statistics, University of Helsinki  
**License:** Mozilla Public License 2.0 (MPL-2.0), Copyright (c) 2026 Olaf Yunus Laitinen Imanov

Arps decline models, model selection with AIC, bootstrap uncertainty and EUR for a whole field.

In [ ]:
# Install wellcast when it is not available yet (for example on Google Colab).
import importlib.util

from IPython import get_ipython

if importlib.util.find_spec("wellcast") is None:
    get_ipython().run_line_magic(
        "pip",
        'install -q "wellcast[all] @ git+https://github.com/olaflaitinen/wellcast.git"',
    )

## The Arps models

For initial rate $q_i$, nominal decline $D_i$ and exponent $b$ (Arps, 1945):

$$q(t) = \frac{q_i}{(1 + b D_i t)^{1/b}}$$

with the exponential ($b = 0$, $q = q_i e^{-D_i t}$) and harmonic ($b = 1$) limits. Cumulative production follows in closed form, and the estimated ultimate recovery (EUR) is the cumulative at the time the rate reaches an economic limit.

In [ ]:
import numpy as np
import pandas as pd

from wellcast.dca import DeclineModel, arps_cumulative, arps_rate, fit_many

t = np.arange(0.0, 3650.0, 30.0)
curves = pd.DataFrame(
    {f"b = {b}": arps_rate(t, 1000.0, 0.8, b) for b in (0.0, 0.5, 1.0)}, index=t
)
curves.iloc[[0, 12, 24, 60, 120]].round(1)

In [ ]:
# Cumulative after ten years for each exponent (bbl):
{b: round(float(arps_cumulative(3650.0, 1000.0, 0.8, b))) for b in (0.0, 0.5, 1.0)}

## Model selection on noisy data

Rates are fitted in log space (errors in rate data are multiplicative) with a robust loss, and the Akaike information criterion picks the model. Here the true exponent is 0.6.

In [ ]:
rng = np.random.default_rng(1)
days = np.arange(0.0, 1500.0)
observed = arps_rate(days, 1200.0, 0.9, 0.6) * rng.lognormal(0.0, 0.05, days.size)
model = DeclineModel().fit(observed, time=days)
print(model)
pd.Series(model.aic_).sort_values().round(1)

## Uncertainty

Resampling the fit residuals and refitting gives a distribution of parameters and EUR. Reserves convention: P90 is the low estimate (exceeded with 90 percent probability), P10 the high one.

In [ ]:
model.bootstrap(300, seed=0)
model.eur_quantiles(econ_limit=10.0).round(0)

In [ ]:
model.plot(pd.Series(observed, index=days), econ_limit=10.0)

## A whole field

`fit_many` fits every well and reports wells that cannot be fitted (here the injector) instead of failing. Any row can be turned back into a model with `DeclineModel.from_params`.

In [ ]:
import wellcast

production = wellcast.make_synthetic_field().production
table = fit_many(production, econ_limit=20.0)
table[["model", "qi", "di", "b", "eur", "remaining", "status"]]

In [ ]:
row = table.loc["P-03"]
rebuilt = DeclineModel.from_params(row["qi"], row["di"], row["b"], start=row["start"])
rebuilt.forecast(365 * 3, econ_limit=20.0).resample("YE")["rate"].mean().round(1)

## Caveats

Arps models assume boundary-dominated flow at constant operating conditions. Choke changes, workovers and water breakthrough break that assumption, which is why the fitted parameters of the synthetic wells differ from the true ones. Fit periods of stable operation, and check the residuals.